In [1]:
!pip install -U transformers
!pip install -U datasets
# !pip install bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 933.0 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 65.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 60.6 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 1.0.0rc2
    Uninstalling huggingface-hub-1.0.0rc2:
      Successfully uninstalled huggingface-hub-1.0.0rc2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.2
    Uninstalling tokenizers-0.21.2:
      Successfully uninstalled tokenizers-0.21.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.3
    Uninstalling transformers-4.53.3:
      Successfully uninstalled transformers-4.53.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. Thi

In [ ]:
from datasets import load_dataset

# ds = load_dataset("AI-MO/NuminaMath-CoT")
ds = load_dataset("5CD-AI/Vietnamese-395k-meta-math-MetaMathQA-gg-translated")

In [ ]:
train_test_split_ds = ds['train'].train_test_split(test_size=0.1, seed=42, shuffle=True)
train_ds = train_test_split_ds['train']
eval_ds = train_test_split_ds['test']

In [ ]:
train_ds = train_ds.to_pandas()
eval_ds = eval_ds.to_pandas()

In [ ]:
train_ds.head()

In [ ]:
eval_ds.head()

In [ ]:
train_ds.info()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 5))
sns.countplot(x='type', data=train_ds, color='lightgreen')
plt.title("Distribution of Types")
plt.xlabel("Type")
plt.ylabel("Count")
plt.show()


In [ ]:
vi_queries = train_ds['query_vi']
query_lens = [len(query) for query in vi_queries]

plt.figure(figsize=(12, 5))
plt.hist(query_lens, bins=50, color='red', alpha=0.5)
plt.title("Distribution of Query Lengths")
plt.xlabel("Query Length")
plt.ylabel("Count")
plt.show()

In [ ]:
response_queries = train_ds['response_vi']
response_lens = [len(query) for query in response_queries]

plt.figure(figsize=(12, 5))
plt.hist(response_lens, bins=50, color='lightblue')
plt.title("Distribution of Response Lengths")
plt.xlabel("Response Length")
plt.ylabel("Count")
plt.show()

## Preprocessing

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Math-1.5B")

In [ ]:
from datasets import Dataset
import pandas as pd

def prepare_data(ds, tokenizer, max_length=512, batch_size=1000):
    """Process dataset efficiently using map"""

    # If input is a DataFrame, convert to Hugging Face Dataset
    if isinstance(ds, pd.DataFrame):
        ds = Dataset.from_pandas(ds)

    def format_and_tokenize(examples):
        formatted_texts = []
        for query, response in zip(examples['query_vi'], examples['response_vi']):
            messages = [
                {"role": "system", "content": "Please reason step by step, and put your final answer."},
                {"role": "user", "content": query},
                {"role": "assistant", "content": response}
            ]
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            formatted_texts.append(text)

        # Tokenize
        tokenized = tokenizer(
            formatted_texts,
            padding='max_length',
            truncation=True,
            max_length=max_length,
        )

        # For causal LM, labels = input_ids
        tokenized['labels'] = tokenized['input_ids']
        return tokenized

    # Use Hugging Face Dataset.map()
    processed_dataset = ds.map(
        format_and_tokenize,
        batched=True,
        batch_size=batch_size,
        num_proc=4,
        remove_columns=ds.column_names,
        desc="Tokenizing dataset"
    )

    return processed_dataset


In [ ]:
eval_ds = prepare_data(eval_ds,tokenizer)

In [ ]:
train_ds = prepare_data(train_ds,tokenizer)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_name = "Qwen/Qwen2.5-Math-1.5B"
device = "cuda" # the device to load the model onto

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.gradient_checkpointing_enable()
model.config.use_cache = False
tokenizer = AutoTokenizer.from_pretrained(model_name)



# prompt = '''
# Consider four points in space:
# A(1,,0,,2), B(4,,1,,0), C(2,,3,,1), D(0,,2,,4).

# Solve the following:

# Compute the **volume** of tetrahedron (ABCD).
# '''

# # CoT
# messages = [
#     {"role": "system", "content": "Please reason step by step, and put your final answer within \\boxed{}."},
#     {"role": "user", "content": prompt}
# ]

# # TIR
# messages = [
#     {"role": "system", "content": "Please integrate natural language reasoning with programs to visualize the problem above, and put your final answer within \\boxed{}."},
#     {"role": "user", "content": prompt}
# ]

# text = tokenizer.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True
# )
# model_inputs = tokenizer([text], return_tensors="pt").to(device)

# generated_ids = model.generate(
#     **model_inputs,
#     max_new_tokens=1000
# )

# generated_ids = [
#     output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
# ]

# response = tokenizer.batch_decode(generated_ids,skip_special_tokens=False)[0]

In [ ]:
from peft import LoraConfig, TaskType
from peft import get_peft_model

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  
    inference_mode=False,
    r=32,  # Reduced from 16 for faster training
    lora_alpha=32,  # Reduced from 32
    lora_dropout=0.05,  # Reduced from 0.1
    target_modules=["q_proj", "v_proj"],  # Target specific modules for efficiency
    bias="none",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


In [ ]:

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

class Config:
    # Training settings
    OUTPUT_DIR = "./results"
    LOGGING_DIR = "./logs"
    EPOCHS = 1
    BATCH_SIZE = 8
    PER_DEVICE_EVAL_BATCH_SIZE = 8
    GRADIENT_ACCUMULATION_STEPS = 16
    LEARNING_RATE = 2e-5
    WEIGHT_DECAY = 0.05
    MAX_GRAD_NORM = 0.4
    WARMUP_RATIO = 0.03
    MAX_STEPS = -1

    # Evaluation and saving
    SAVE_STRATEGY = "steps"
    EVAL_STRATEGY = "steps"
    EVAL_STEPS = 700
    SAVE_STEPS = 700
    SAVE_TOTAL_LIMIT = 2
    LOGGING_STEPS = 10

    # Mixed precision
    FP16 = True
    BF16 = False

    # Other settings
    GROUP_BY_LENGTH = True
    LR_SCHEDULER_TYPE = "constant_with_warmup"
    REPORT_TO = "tensorboard"
    LOAD_BEST_MODEL_AT_END = True
    METRIC_FOR_BEST_MODEL = "eval_loss"
    GREATER_IS_BETTER = False
    EVAL_ACCUMULATION_STEPS = 8

cfg = Config()

In [ ]:
from transformers import (
    TrainingArguments,
    Trainer,
    TrainerCallback,
    DataCollatorForLanguageModeling
)
from transformers import EarlyStoppingCallback
# Set padding token if not set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

training_args = TrainingArguments(
    output_dir=cfg.OUTPUT_DIR,
    num_train_epochs=cfg.EPOCHS,
    per_device_train_batch_size=cfg.BATCH_SIZE,
    per_device_eval_batch_size=cfg.PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=cfg.GRADIENT_ACCUMULATION_STEPS,
    save_strategy=cfg.SAVE_STRATEGY,
    eval_strategy=cfg.EVAL_STRATEGY,
    eval_steps=cfg.EVAL_STEPS,
    save_steps=cfg.SAVE_STEPS,
    save_total_limit=cfg.SAVE_TOTAL_LIMIT,
    logging_steps=cfg.LOGGING_STEPS,
    learning_rate=cfg.LEARNING_RATE,
    weight_decay=cfg.WEIGHT_DECAY,
    fp16=cfg.FP16,
    bf16=cfg.BF16,
    max_grad_norm=cfg.MAX_GRAD_NORM,
    max_steps=cfg.MAX_STEPS,
    warmup_ratio=cfg.WARMUP_RATIO,
    group_by_length=cfg.GROUP_BY_LENGTH,
    lr_scheduler_type=cfg.LR_SCHEDULER_TYPE,
    report_to=cfg.REPORT_TO,
    logging_dir=cfg.LOGGING_DIR,
    load_best_model_at_end=cfg.LOAD_BEST_MODEL_AT_END,
    metric_for_best_model=cfg.METRIC_FOR_BEST_MODEL,
    greater_is_better=cfg.GREATER_IS_BETTER,
    eval_accumulation_steps=cfg.EVAL_ACCUMULATION_STEPS,
    dataloader_num_workers=4,  
    dataloader_pin_memory=True,  
    ddp_find_unused_parameters=False,
    remove_unused_columns=False,
    gradient_checkpointing=True,   
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,  
    eval_dataset=eval_ds,   
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]
)


In [ ]:
trainer.train()

# Agent

In [27]:
from datasets import load_dataset

# ds = load_dataset("AI-MO/NuminaMath-CoT")
ds = load_dataset("5CD-AI/Vietnamese-395k-meta-math-MetaMathQA-gg-translated")

In [28]:
train_test_split_ds = ds['train'].train_test_split(test_size=0.1, seed=42, shuffle=True)
train_ds = train_test_split_ds['train']
eval_ds = train_test_split_ds['test']

In [4]:
train_ds = train_ds.to_pandas()
eval_ds = eval_ds.to_pandas()

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Math-1.5B")

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# print(tokenizer.chat_template)

In [21]:
import pandas as pd
from datasets import Dataset
def prepare_data(ds, tokenizer, max_length=512, batch_size=1000):
    """Process dataset efficiently using map"""

    # If input is a DataFrame, convert to Hugging Face Dataset
    if isinstance(ds, pd.DataFrame):
        ds = Dataset.from_pandas(ds)

    def format_and_tokenize(examples):
        formatted_texts = []
        for query, response in zip(examples['query_vi'], examples['response_vi']):
            messages = [
                {"role": "system", "content": "Please reason step by step, and give your final answer."},
                {"role": "user", "content": query},
                {"role": "assistant", "content": response}
            ]
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
                tools= tools
            )
            formatted_texts.append(text)

        # Tokenize
        tokenized = tokenizer(
            formatted_texts,
            padding='max_length',
            truncation=True,
            max_length=max_length,
        )

        # For causal LM, labels = input_ids
        tokenized['labels'] = tokenized['input_ids']
        return tokenized

    # Use Hugging Face Dataset.map()
    processed_dataset = ds.map(
        format_and_tokenize,
        batched=True,
        batch_size=batch_size,
        num_proc=4,
        remove_columns=ds.column_names,
        desc="Tokenizing dataset"
    )

    return processed_dataset

# Tools

In [7]:
import json
import numexpr

def Calculator(query: str) -> str:
    """NumExpr based calculator, which is a fast numerical expression evaluator for NumPy.
    codes adapted from: https://github.com/ernie-research/Tool-Augmented-Reward-Model/blob/main/src/tools/calculator.py#L1
    
    Example usage:
    answer = Calculator("2+3")
    print(answer) # 5.0

    Args:
        query (str): a math formula, supports +, -, * amd /

    Returns:
        str: calculated answer
    """
    try:
        tool_response = str(numexpr.evaluate(query))
    except Exception as e:
        print(e)
        tool_response = "Error: failed to calculate {}.".format(query)
    return tool_response




In [8]:
tools = [
    {
        "name": "Calculator",
        "description": "Evaluate a numeric expression given as a string, using + - * / and parentheses.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A numeric expression, e.g. '4+5*(2-1)'"
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "Wikipedia_retriever",
        "description": "Return relevant Wikipedia document text for a query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"},
                "k": {"type": "integer", "description": "Top-k results to return", "default": 1}
            },
            "required": ["query"]
        }
    }
]


In [34]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'Please reason step by step, and put your final answer within \\boxed{}.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nPlease reason step by step, and

In [37]:
query = train_ds["query_vi"][0]   # assumed defined earlier
response = train_ds["response_vi"][0]

messages = [
    {"role": "system", "content": "Please reason step by step, and give your final answer."},
    {"role": "user", "content": query},
    # Example assistant thought + tool call
    # {
    #     "role": "assistant",
    #     "content": "I'll compute the numeric expression first.",
    #     "tool_calls": [
    #         {
    #             # your template supports tool_call.function or tool_call.name
    #             "function": {
    #                 "name": "Calculator",
    #                 "arguments": {"expression": "4+5"}
    #             }
    #         }
    #     ]
    # },
    # tool response recorded in the dataset
    # {"role": "tool", "content": "9.0"},
    # assistant final response (teacher / human-provided)
    {"role": "assistant", "content": response}
]

# Format for model input using your tokenizer template:
formatted = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False,
    tools=tools
)
print(formatted)  # inspect the result to be sure it matches expectations


<|im_start|>system
Please reason step by step, and give your final answer.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"name": "Calculator", "description": "Evaluate a numeric expression given as a string, using + - * / and parentheses.", "parameters": {"type": "object", "properties": {"expression": {"type": "string", "description": "A numeric expression, e.g. '4+5*(2-1)'"}}, "required": ["expression"]}}
{"name": "Wikipedia_retriever", "description": "Return relevant Wikipedia document text for a query.", "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "Search query"}, "k": {"type": "integer", "description": "Top-k results to return", "default": 1}}, "required": ["query"]}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <functi

In [22]:
eval_ds = prepare_data(eval_ds, tokenizer)
train_ds = prepare_data(train_ds, tokenizer)

Tokenizing dataset (num_proc=4):   0%|          | 0/39500 [00:00<?, ? examples/s]

Tokenizing dataset (num_proc=4):   0%|          | 0/355500 [00:00<?, ? examples/s]

## Utils

In [24]:
import re

def extract_answer(response):
    try:
        answer = response.split("\nAnswer:")[1].strip()
    except Exception as e:
        # print(e)
        answer = response
    return answer

def extract_tool_usage(llm_text: str):
    """Extract tool usage patterns such as ToolName[arg] from the model output.
       Returns list of strings like 'Calculator[1+2]' """
    try:
        action = llm_text.split("\nAction:")[1].split("\nRationale:")[0]
    except Exception:
        action = llm_text
    return re.findall(r'\w+\[[^\[\]]+\]', action)


def regex_math_answer(model_answer, gt_answer_list):
    # extract the last number as model answer if matched
    # regex pattern adapted from: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/lm_eval/tasks/gsm8k/gsm8k-cot-zeroshot.yaml#L41
    try:
        pattern = "(-?[$0-9.,]{2,})|(-?[0-9]+)"
        matched = re.findall(pattern, model_answer)
        if matched:
            model_answer = matched[-1][0] if matched[-1][0] else matched[-1][1]
    except Exception as e:
        # print(e)
        pass
    return model_answer, gt_answer_list


def parse_tool_call_from_text(text: str):
    """Parse first <tool_call>{...}</tool_call> if present and return dict or None."""
    m = re.search(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", text, flags=re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(1))
    except Exception as e:
        print("Failed to parse tool_call JSON:", e)
        return None




In [ ]:
class ToolUseAgent:
    def __init__(
        self,
        model,
        tokenizer,
        tools_metadata=None,
        generation_cfg=None,
    ):

        
        self.model = model
        self.tokenizer = tokenizer
        self.tools = tools_metadata or []
    
        # generation defaults
        self.generation_cfg = generation_cfg or {
            "max_new_tokens": 256,
            "do_sample": False,
            "temperature": 0.0,
            "top_p": 0.95,
        }

    def invoke_tool(self, func_str: str) -> str:
        
        if not isinstance(func_str, str) or "[" not in func_str:
            return "Error: invalid tool invocation format."

        tool_name = func_str.split("[", 1)[0]
        query = func_str.split("[", 1)[1].rsplit("]", 1)[0]

        if tool_name.lower() in ("calculator", "calculator"):
            return Calculator(query)
        elif tool_name.lower() in ("wikipediasearch", "wikipedia_retriever", "wikipediaretriever"):
            return WikipediaRetriever(query)
        else:
            return f"Error: tool `{tool_name}` not found."


    def call_llm(self, conversations: list, add_generation_prompt=True):
    
        # render chat template (this produces a string)
        prompt_text = self.tokenizer.apply_chat_template(
            conversations,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            tools=self.tools if self.tools else None,
        )

        # tokenize to tensors
        encoded = self.tokenizer(prompt_text, return_tensors="pt")
        input_ids = encoded["input_ids"].to(self.device)
        attention_mask = encoded.get("attention_mask", None)
        if attention_mask is not None:
            attention_mask = attention_mask.to(self.device)

        # generation arguments
        gen_kwargs = dict(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=self.generation_cfg.get("max_new_tokens", 256),
            do_sample=self.generation_cfg.get("do_sample", False),
            temperature=self.generation_cfg.get("temperature", 0.0),
            top_p=self.generation_cfg.get("top_p", 0.95),
            pad_token_id=self.tokenizer.eos_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )

        # generate
        outputs = self.model.generate(**gen_kwargs)
        # outputs: (batch, seq_len_total). We only generated 1 sample so batch=1
        generated = outputs[0, input_ids.shape[-1]:].cpu().numpy()
        decoded = self.tokenizer.decode(generated, skip_special_tokens=True).strip()
        return decoded

    
    def inference(self, question: str, prompt_type="default"):
       
        system_prompt = "Please reason step by step, and put your final answer within \\boxed{}."

        conversations = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ]

        llm_response = self.call_llm(conversations, add_generation_prompt=True)
        conversations.append({"role": "assistant", "content": llm_response})

        # First check for explicit <tool_call> JSON (preferred)
        tool_call = parse_tool_call_from_text(llm_response)
        # Also support simple patterns like ToolName[query]
        tools_simple = extract_tool_usage(llm_response)

        while tool_call is not None or len(tools_simple) > 0:
            if tool_call is not None:
                name = tool_call.get("name")
                args = tool_call.get("arguments", {})
                if "expression" in args:
                    invocation = f"{name}[{args['expression']}]"
                elif "query" in args:
                    invocation = f"{name}[{args['query']}]"
                else:
                    invocation = f"{name}[{json.dumps(args)}]"
                tool_res = self.invoke_tool(invocation)
                tool_text = f"Response from tool {invocation}: {tool_res}"
                conversations.append({"role": "user", "content": tool_text})
                llm_response = self.call_llm(conversations, add_generation_prompt=True)
                conversations.append({"role": "assistant", "content": llm_response})
            else:
            
                for func in tools_simple:
                    tool_res = self.invoke_tool(func)
                    tool_text = f"Response from tool {func}: {tool_res}"
                    conversations.append({"role": "tool", "content": tool_text})
                llm_response = self.call_llm(conversations, add_generation_prompt=True)
                conversations.append({"role": "assistant", "content": llm_response})

            tool_call = parse_tool_call_from_text(llm_response)
            tools_simple = extract_tool_usage(llm_response)

        return conversations, llm_response
    

    def train(self, train_dataset, eval_dataset, cfg):
        
        self.model.gradient_checkpointing_enable()
        try:
            self.model.config.use_cache = False
        except Exception:
            pass

        data_collator = DataCollatorForCompletionOnlyLM(tokenizer=self.tokenizer)

        training_args = TrainingArguments(
            output_dir=cfg.OUTPUT_DIR,
            num_train_epochs=cfg.EPOCHS,
            per_device_train_batch_size=cfg.BATCH_SIZE,
            per_device_eval_batch_size=getattr(cfg, "PER_DEVICE_EVAL_BATCH_SIZE", cfg.BATCH_SIZE),
            gradient_accumulation_steps=cfg.GRADIENT_ACCUMULATION_STEPS,
            learning_rate=cfg.LEARNING_RATE,
            weight_decay=getattr(cfg, "WEIGHT_DECAY", 0.01),
            warmup_ratio=getattr(cfg, "WARMUP_RATIO", 0.1),
            lr_scheduler_type=getattr(cfg, "LR_SCHEDULER_TYPE", "cosine"),
            fp16=getattr(cfg, "FP16", False),
            evaluation_strategy=getattr(cfg, "EVAL_STRATEGY", "steps"),
            eval_steps=getattr(cfg, "EVAL_STEPS", 1000),
            save_strategy=getattr(cfg, "SAVE_STRATEGY", "steps"),
            save_steps=getattr(cfg, "SAVE_STEPS", 1000),
            save_total_limit=getattr(cfg, "SAVE_TOTAL_LIMIT", 3),
            logging_steps=getattr(cfg, "LOGGING_STEPS", 50),
            logging_dir=getattr(cfg, "LOGGING_DIR", None),
            load_best_model_at_end=getattr(cfg, "LOAD_BEST_MODEL_AT_END", True),
            metric_for_best_model=getattr(cfg, "METRIC_FOR_BEST_MODEL", "eval_loss"),
            remove_unused_columns=False,
            gradient_checkpointing=True,
        )

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            tokenizer=self.tokenizer,
            data_collator=data_collator,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )

        trainer.train()
        trainer.save_model(training_args.output_dir)